# F04 - Data cleaning, missing values, categorical features, and normalization

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** F03

Real tables arrive with missing values, inconsistent labels, and duplicates.
This notebook turns a deliberately messy synthetic student table into a model-
ready table. Every transformation is fit inside a scikit-learn pipeline so it
can be reused on validation and test data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Inspect before cleaning

The table has missing numeric values, inconsistent category spellings, and
duplicate rows. Count each problem before deciding what to do. "Clean the data"
is not a plan; "drop exact duplicates, normalize course names, and impute three
numeric columns with the training median" is a plan.

In [2]:
def make_messy_students(n: int = 320, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    study = rng.gamma(2.0, 2.0, n).clip(0, 20)
    attendance = rng.uniform(40, 100, n)
    prior = rng.normal(65, 12, n).clip(0, 100)
    sleep = rng.normal(7, 1.2, n).clip(3, 10)
    tutoring = rng.choice(["yes", "no"], n, p=[0.35, 0.65])
    course = rng.choice(["Math", "Physics", "CS"], n, p=[0.4, 0.3, 0.3])
    logit = -11.5 + 0.3 * study + 0.055 * attendance + 0.075 * prior + 1.1 * (tutoring == "yes")
    passed = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    frame = pd.DataFrame(
        {
            "study_hours": study.round(2),
            "attendance": attendance.round(1),
            "prior_score": prior.round(1),
            "sleep_hours": sleep.round(2),
            "tutoring": tutoring,
            "course": course,
            "passed": passed,
        }
    )
    for column, fraction in {"attendance": 0.08, "prior_score": 0.06, "sleep_hours": 0.10}.items():
        frame.loc[rng.choice(n, size=int(n * fraction), replace=False), column] = np.nan
    frame.loc[rng.choice(n, size=18, replace=False), "course"] = rng.choice(
        [" math ", "MATH", "physics", "CS "], 18
    )
    frame.loc[rng.choice(n, size=14, replace=False), "tutoring"] = rng.choice(
        ["Yes", "NO", "yes "], 14
    )
    return pd.concat([frame, frame.sample(8, random_state=seed)], ignore_index=True)


df = make_messy_students()
print("shape:", df.shape)
print("missing values:")
print(df.isna().sum().to_string())
print("duplicate rows:", int(df.duplicated().sum()))
print("raw course values:", sorted(df["course"].unique()))
assert df.isna().sum().sum() > 0
assert df.duplicated().sum() > 0

shape: (328, 7)
missing values:
study_hours     0
attendance     26
prior_score    19
sleep_hours    34
tutoring        0
course          0
passed          0
duplicate rows: 8
raw course values: [' math ', 'CS', 'CS ', 'MATH', 'Math', 'Physics', 'physics']


## 2. Remove exact duplicates and normalize labels

Exact duplicates are usually accidental. Remove them before splitting so the
same row cannot appear in both training and test data. Category spellings are
normalized with a mapping; values outside the mapping become `"Unknown"` rather
than silently disappearing.

In [3]:
clean = df.drop_duplicates().copy()
clean["course"] = (
    clean["course"].str.strip().str.lower()
    .map({"math": "Math", "physics": "Physics", "cs": "CS"})
    .fillna("Unknown")
)
clean["tutoring"] = clean["tutoring"].str.strip().str.lower().map(
    {"yes": "yes", "no": "no"}
).fillna("unknown")
print("rows after duplicate removal:", len(clean))
print(clean["course"].value_counts().to_string())
assert len(clean) < len(df)
assert set(clean["course"]) <= {"Math", "Physics", "CS", "Unknown"}

rows after duplicate removal: 320
course
Math       128
CS         105
Physics     87


## 3. Numeric imputation

The median is robust to skew and outliers. The mean is sensitive to extreme
values. `SimpleImputer` learns one value per column from the training data; it
does not look at the test set. That is the difference between imputation and
leakage.

In [4]:
numeric_columns = ["study_hours", "attendance", "prior_score", "sleep_hours"]
categorical_columns = ["tutoring", "course"]

numeric_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_columns),
        ("cat", categorical_pipe, categorical_columns),
    ]
)
model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]
)
print(model)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['study_hours', 'attendance',
                                                   'prior_score',
                                                   'sleep_hours']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                   

## 4. Compare dropping rows with imputation

Dropping every row with a missing value is simple but throws away information and
can bias the sample. The comparison below uses the same 5-fold split for both
strategies. The imputed model should use more rows; its score may be similar or
better.

In [5]:
y = clean["passed"].to_numpy()
X = clean[numeric_columns + categorical_columns]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

imputed_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print("imputed folds:", np.round(imputed_scores, 3))
print("imputed mean:", round(imputed_scores.mean(), 3))

complete = clean.dropna().copy()
drop_scores = cross_val_score(
    model, complete[numeric_columns + categorical_columns], complete["passed"], cv=cv, scoring="accuracy"
)
print("complete-case rows:", len(complete), "of", len(clean))
print("complete-case mean:", round(drop_scores.mean(), 3))
assert imputed_scores.mean() > 0.5
assert len(complete) < len(clean)

imputed folds: [0.766 0.781 0.781 0.781 0.844]
imputed mean: 0.791
complete-case rows: 248 of 320
complete-case mean: 0.77


## 5. Fit the final preprocessing and inspect the matrix

`fit_transform` on the full cleaned table is acceptable here because this is a
demonstration, not an evaluation. In a real experiment you fit on training data
only, then call `transform` on validation and test data. The output shape tells
you how many encoded columns the model sees.

In [6]:
X_transformed = preprocess.fit_transform(clean[numeric_columns + categorical_columns])
print("transformed shape:", X_transformed.shape)
print("first row:", np.round(X_transformed[0], 3))
assert X_transformed.shape[0] == len(clean)
assert np.isfinite(X_transformed).all()

processed = clean.copy()
processed[numeric_columns] = preprocess.named_transformers_["num"].named_steps["impute"].transform(
    clean[numeric_columns]
)
assert processed[numeric_columns].isna().sum().sum() == 0
processed_path = OUTPUT_DIR / "students_clean.csv"
processed.to_csv(processed_path, index=False)
print("wrote", processed_path)

transformed shape: (320, 9)
first row: [-0.03  -0.892  0.8   -0.026  0.     1.     0.     0.     1.   ]


wrote outputs\students_clean.csv


## Common decisions and their trade-offs

| Decision | Benefit | Risk |
| --- | --- | --- |
| Median imputation | Robust, keeps rows | Shrinks variance; hides missingness |
| Add missing indicator | Preserves "was missing" signal | More columns, harder to explain |
| One-hot encoding | No false ordering | Wide matrices for high cardinality |
| Ordinal encoding | Compact | Implies an order that may not exist |
| Standard scaling | Helps linear models and neural nets | Must be fit on training data only |
| Dropping rows | Simple | Can bias the sample and lose rare cases |

**Next:** F05 - Train, validation, and test splits.